# Part 1: The Coding Implementation 

Library Usage

In [ ]:
import nltk
import random
import math

## Step 1: Data Acquisition and Splitting

Use the Natural Language Toolkit (nltk) to download the Brown Corpus using the universal tagset.

In [20]:
import nltk
import random

# Download required NLTK data
nltk.download("brown")
nltk.download("universal_tagset")

from nltk.corpus import brown

[nltk_data] Downloading package brown to /home/super/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/super/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


### A. import and split data 

In [21]:
# TODO : Load the corpus and split it into 80% training data and 20% testing data.

# Load Brown Corpus sentences with universal POS tags
sentences = brown.tagged_sents(tagset="universal")

# Convert to a normal list so we can shuffle and split it
sentences = list(sentences)

print("Number of sentences:", len(sentences))
print("First sentence:")
print(sentences[0])

Number of sentences: 57340
First sentence:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [22]:
# Shuffle the sentences so training and testing data are mixed fairly
random.seed(42)
random.shuffle(sentences)

# 80% for training, 20% for testing
split_index = int(0.8 * len(sentences))

train_data = sentences[:split_index]
test_data = sentences[split_index:]

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 45872
Testing sentences: 11468


In [23]:
# Check one training sentence
print("Example training sentence:")
print(train_data[0])

# Check sentence length
print("\nNumber of words in this sentence:", len(train_data[0]))

Example training sentence:
[('He', 'PRON'), ('let', 'VERB'), ('her', 'PRON'), ('tell', 'VERB'), ('him', 'PRON'), ('all', 'PRT'), ('about', 'ADP'), ('the', 'DET'), ('church', 'NOUN'), ('.', '.')]

Number of words in this sentence: 10


## Step 2: Parameter Estimation & Laplace Smoothing

Train your HMM by estimating the initial ($\pi$), transition ($A$), and emission ($B$) probabilities using Maximum Likelihood Estimation (MLE) based on frequency counts in your training set.

To prevent zero-probabilities when encountering unseen words during testing, apply **Laplace (Add-$\alpha$) Smoothing** to your emission matrix. The smoothed probability of emitting word $o_t$ given tag $s_i$ is:

$$P_{\text{smoothed}}(o_t|s_i) = \frac{\text{Count}(o_t \text{ tagged as } s_i) + \alpha}{\text{Count}(s_i) + \alpha|V|}$$

Where $\alpha = 1.0$ and $|V|$ is the total number of unique words in your training vocabulary. Ensure you create a specific dictionary key (e.g., `"<OOV>"`) to store the probability mass for unseen words.



In [24]:
from collections import Counter, defaultdict

# Smoothing value required by the project
alpha = 1.0

# Special token for unseen words
OOV_TOKEN = "<OOV>"

# Helper function: make words lowercase to reduce vocabulary size
def normalize_word(word):
    return word.lower()

# Count how often each tag starts a sentence
initial_counts = Counter()

# Count tag-to-tag transitions
# Example: transition_counts["DET"]["NOUN"]
transition_counts = defaultdict(Counter)

# Count tag-to-word emissions
# Example: emission_counts["NOUN"]["dog"]
emission_counts = defaultdict(Counter)

# Count total number of each tag
tag_counts = Counter()

# Store all unique words and tags
vocab = set()
tags = set()

In [25]:
# Count initial tags, transitions, and emissions from training data

for sentence in train_data:
    # Skip empty sentences, just in case
    if len(sentence) == 0:
        continue

    # Get the first tag of the sentence
    first_word, first_tag = sentence[0]
    initial_counts[first_tag] += 1

    # Loop through every word-tag pair in the sentence
    previous_tag = None

    for word, tag in sentence:
        # Normalize word
        word = normalize_word(word)

        # Save vocabulary and tag
        vocab.add(word)
        tags.add(tag)

        # Count how many times this tag appears
        tag_counts[tag] += 1

        # Count emission: tag -> word
        emission_counts[tag][word] += 1

        # Count transition: previous_tag -> current_tag
        if previous_tag is not None:
            transition_counts[previous_tag][tag] += 1

        # Update previous tag
        previous_tag = tag

# Convert sets to sorted lists for stable order
vocab = sorted(list(vocab))
tags = sorted(list(tags))

print("Number of unique words:", len(vocab))
print("Number of POS tags:", len(tags))
print("POS tags:", tags)

Number of unique words: 45153
Number of POS tags: 12
POS tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']


In [ ]:
# Estimate initial probabilities pi

pi = {}

total_sentences = len(train_data)

for tag in tags:
    pi[tag] = initial_counts[tag] / total_sentences

print("Initial probability:")
for tag in tags:
    print(tag, ":", pi[tag])

Initial probability example:
. : 0.0870247645622602
ADJ : 0.03483606557377049
ADP : 0.12371381234740146
ADV : 0.09042553191489362
CONJ : 0.0492893268224625
DET : 0.21365974886641087
NOUN : 0.14150244157656086
NUM : 0.016437042204394837
PRON : 0.1595962678758284
PRT : 0.03712504359958144
VERB : 0.04582316009766306
X : 0.0005667945587722358


In [ ]:
# Estimate transition probabilities A

A = {}

for prev_tag in tags:
    A[prev_tag] = {}

    # Total transitions starting from prev_tag
    total_transitions_from_prev_tag = sum(transition_counts[prev_tag].values())

    for next_tag in tags:
        if total_transitions_from_prev_tag == 0:
            A[prev_tag][next_tag] = 0.0
        else:
            A[prev_tag][next_tag] = (
                transition_counts[prev_tag][next_tag] / total_transitions_from_prev_tag
            )

print("Transition probability:")
print("P(NOUN -> VERB) =", A["NOUN"]["VERB"], "(VERB|NOUN)")

Transition probability:
P(NOUN -> VERB) = 0.15922801845330714


In [32]:
# Estimate emission probabilities B using Laplace smoothing

B = {}

# We add one extra vocabulary item for OOV words
vocab_size_with_oov = len(vocab) + 1

for tag in tags:
    B[tag] = {}

    # Denominator for Laplace smoothing
    denominator = tag_counts[tag] + alpha * vocab_size_with_oov

    # Probability for each known word
    for word in vocab:
        B[tag][word] = (
            emission_counts[tag][word] + alpha
        ) / denominator

    # Probability for unseen words
    B[tag][OOV_TOKEN] = alpha / denominator

print("Emission probability:")
print("P(dog | NOUN) =", B["NOUN"].get("dog", B["NOUN"][OOV_TOKEN]))
print("P(<OOV> | NOUN) =", B["NOUN"][OOV_TOKEN])

Emission probability:
P(dog | NOUN) = 0.00020324744246968225
P(<OOV> | NOUN) = 3.763841527216338e-06


In [34]:
# Quick probability checks

print("Sum of pi probabilities:", sum(pi.values()))

example_tag = "NOUN"

print("Sum of emission probabilities for", example_tag, ":")
print(sum(B[example_tag].values()))

example_prev_tag = "NOUN"

print("Sum of transition probabilities from", example_prev_tag, ":")
print(sum(A[example_prev_tag].values()))

Sum of pi probabilities: 1.0
Sum of emission probabilities for NOUN :
1.0000000000003673
Sum of transition probabilities from NOUN :
1.0


## Step 3: The Log-Space Viterbi Decoder

Because standard probabilities are strictly between 0 and 1, multiplying them together for a normal-length sentence results in infinitesimally small numbers (Underflow). Transform the recursive Viterbi step into log-space:

$$\log v_t(j) = \max_{i=1}^N \left[ \log v_{t-1}(i) + \log P(s_j|s_i) \right] + \log P(o_t|s_j)$$

*Hint: $\log(0)$ is mathematically undefined. Add a tiny constant (epsilon, $10^{-12}$) to all probabilities before taking the logarithm to avoid math domain errors.*


In [35]:
import math

# Tiny value to avoid log(0)
EPSILON = 1e-12

def get_emission_probability(tag, word, B):
    """
    Get P(word | tag).
    If the word is unseen, use the <OOV> probability.
    """
    word = normalize_word(word)

    if word in B[tag]:
        return B[tag][word]
    else:
        return B[tag][OOV_TOKEN]


def log_viterbi(sentence, tags, A, B, pi):
    """
    Predict the best POS tag sequence for a sentence using log-space Viterbi.

    sentence: list of words
    tags: list of all possible POS tags
    A: transition probabilities
    B: emission probabilities
    pi: initial probabilities
    """

    # If sentence is empty, return empty tag list
    if len(sentence) == 0:
        return []

    # viterbi[t][tag] stores the best log-probability
    # for position t ending with this tag
    viterbi = []

    # backpointer[t][tag] stores the best previous tag
    # that leads to this current tag
    backpointer = []

    # -------------------------
    # 1. Initialization step
    # -------------------------
    first_word = sentence[0]

    viterbi.append({})
    backpointer.append({})

    for tag in tags:
        emission_prob = get_emission_probability(tag, first_word, B)

        # log P(tag starts sentence) + log P(first_word | tag)
        viterbi[0][tag] = math.log(pi.get(tag, 0.0) + EPSILON) + math.log(emission_prob + EPSILON)

        # First word has no previous tag
        backpointer[0][tag] = None

    # -------------------------
    # 2. Forward pass
    # -------------------------
    for t in range(1, len(sentence)):
        word = sentence[t]

        viterbi.append({})
        backpointer.append({})

        for current_tag in tags:
            emission_prob = get_emission_probability(current_tag, word, B)

            best_previous_tag = None
            best_log_prob = float("-inf")

            for previous_tag in tags:
                transition_prob = A[previous_tag].get(current_tag, 0.0)

                # Previous best score
                # + log P(current_tag | previous_tag)
                # + log P(word | current_tag)
                log_prob = (
                    viterbi[t - 1][previous_tag]
                    + math.log(transition_prob + EPSILON)
                    + math.log(emission_prob + EPSILON)
                )

                if log_prob > best_log_prob:
                    best_log_prob = log_prob
                    best_previous_tag = previous_tag

            viterbi[t][current_tag] = best_log_prob
            backpointer[t][current_tag] = best_previous_tag

    # -------------------------
    # 3. Backtracking step
    # -------------------------

    # Find the best final tag
    best_final_tag = max(viterbi[-1], key=viterbi[-1].get)

    best_tag_sequence = [best_final_tag]

    # Move backward from the last word to the first word
    for t in range(len(sentence) - 1, 0, -1):
        best_previous_tag = backpointer[t][best_tag_sequence[-1]]
        best_tag_sequence.append(best_previous_tag)

    # Reverse because we collected tags from end to start
    best_tag_sequence.reverse()

    return best_tag_sequence

## Step 4: Evaluation

Write an evaluation loop to process the 20% test sentences and compute your total word-level accuracy.

$$\text{Accuracy} = \frac{\text{Total Correctly Tagged Words}}{\text{Total Words in Test Set}}$$


In [28]:
# Step 4: Evaluation
# Goal: compute word-level accuracy on the 20% test set

total_correct = 0  # Count how many words get the correct POS tag
total_words = 0    # Count total words in the test set

# Go through each sentence in the test data
for sentence in test_data:
    if len(sentence) == 0:
        continue

    # Separate words and true tags
    words = [word for word, tag in sentence]
    true_tags = [tag for word, tag in sentence]

    # Predict tags using our HMM model
    predicted_tags = log_viterbi(words, tags, A, B, pi)

    # Compare predicted tags with true tags
    for true_tag, predicted_tag in zip(true_tags, predicted_tags):
        if true_tag == predicted_tag:
            total_correct += 1

        total_words += 1

# Calculate accuracy
accuracy = total_correct / total_words

print("Total correctly tagged words:", total_correct)
print("Total words in test set:", total_words)
print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

NameError: name 'A' is not defined